In [1]:
import trimesh
import numpy as np
import glob
import os

In [2]:
def compute_global_min_max(file_list):
    """
    Compute the global minimum and maximum vertex values across all shapes.
    """
    global_min = np.inf
    global_max = -np.inf
    global_avg = 0

    count = 0
    for file in file_list:
        mesh = trimesh.load(file)
        vertices = mesh.vertices

        # Update global min and max
        global_min = min(global_min, vertices.min())
        global_max = max(global_max, vertices.max())
        global_avg += vertices.mean()

        count += 1
        if count % 50 == 0:
            print(f"Processed {count} shapes.")

    global_avg /= len(file_list)
    
    return global_min, global_max, global_avg

def scale_mesh_to_uniform_range(mesh, global_min, global_max, target_min=-0.90, target_max=0.90):
    """
    Scale a mesh such that the vertex coordinates are mapped to a global range [-0.95, 0.95],
    while preserving the shape uniformly across all dimensions.
    """
    # Compute the global range and the target range
    global_range = global_max - global_min
    target_range = target_max - target_min

    # Compute the scaling factor based on the largest dimension range
    scaling_factor = target_range / global_range

    # Scale the vertices uniformly
    vertices = mesh.vertices
    scaled_vertices = (vertices - global_min) * scaling_factor + target_min
    print(f"Scaled vertices to range [{scaled_vertices.min()}, {scaled_vertices.max()}]")

    # Update the mesh vertices
    mesh.vertices = scaled_vertices

    return mesh


In [3]:
# Input files
input_files = glob.glob("/home/jakaria/CALSNIC/calsnic_pial_surface/mesh_dataset/pial_surface/obj_files/*.obj")  
output_folder = "/home/jakaria/CALSNIC/calsnic_pial_surface/mesh_dataset/pial_surface/scaled_obj_files/" 

# Create output folder if it doesn't exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

In [4]:
# Compute global min and max
global_min, global_max, global_avg = compute_global_min_max(input_files)
print(f"Global min: {global_min}, Global max: {global_max}, Global avg: {global_avg}")

Processed 50 shapes.
Processed 100 shapes.
Processed 150 shapes.
Processed 200 shapes.
Global min: -109.70689392, Global max: 101.6706543, Global avg: -1.8457190826288987


In [5]:
# Process and save scaled meshes
count = 0
for input_file in input_files:
    mesh = trimesh.load(input_file)
    scaled_mesh = scale_mesh_to_uniform_range(mesh, global_min, global_max)

    # Save the scaled mesh
    output_file = f"{output_folder}{input_file.split('/')[-1]}"
    print(f"Saving scaled mesh to: {output_file}")
    scaled_mesh.export(output_file)
    count += 1
    if count % 50 == 0:
        print(f"Processed {count} meshes.")

print("All meshes scaled and saved successfully.")

Scaled vertices to range [-0.6918392034701595, 0.8249788364491041]
Saving scaled mesh to: /home/jakaria/CALSNIC/calsnic_pial_surface/mesh_dataset/pial_surface/scaled_obj_files/CALSNIC2_QUE_C029.obj
Scaled vertices to range [-0.7555106718230437, 0.667190104510145]
Saving scaled mesh to: /home/jakaria/CALSNIC/calsnic_pial_surface/mesh_dataset/pial_surface/scaled_obj_files/CALSNIC2_EDM_C080.obj
Scaled vertices to range [-0.8551217479818302, 0.7435327949703666]
Saving scaled mesh to: /home/jakaria/CALSNIC/calsnic_pial_surface/mesh_dataset/pial_surface/scaled_obj_files/CALSNIC2_MON_C008.obj
Scaled vertices to range [-0.8127087306604772, 0.6567595854859328]
Saving scaled mesh to: /home/jakaria/CALSNIC/calsnic_pial_surface/mesh_dataset/pial_surface/scaled_obj_files/CALSNIC1_VAN_C011.obj
Scaled vertices to range [-0.7899117574692437, 0.6383322075131977]
Saving scaled mesh to: /home/jakaria/CALSNIC/calsnic_pial_surface/mesh_dataset/pial_surface/scaled_obj_files/CALSNIC2_EDM_C050.obj
Scaled vert

In [6]:
# Compute global min and max
input_files_scaled = glob.glob("/home/jakaria/CALSNIC/calsnic_pial_surface/mesh_dataset/pial_surface/scaled_obj_files/*.obj")  
global_min, global_max, global_avg = compute_global_min_max(input_files_scaled)
print(f"Global min: {global_min}, Global max: {global_max}, Global avg: {global_avg}")

Processed 50 shapes.
Processed 100 shapes.
Processed 150 shapes.
Processed 200 shapes.
Global min: -0.9, Global max: 0.9, Global avg: 0.018499227293591183


In [2]:
import pandas as pd
import torch
import os

# Read the CSV file
csv_path = "/home/jakaria/final_classification_dataset_femur_original/labels/binary_labels.csv"
df = pd.read_csv(csv_path, names=['patient_id', 'label', 'label_name', 'kl_grade'], skiprows=1)

# Create labels dictionary
labels = {}
for _, row in df.iterrows():
    patient_id = str(row['patient_id'])
    label = float(row['label'])
    labels[patient_id] = label

print(f"Created labels for {len(labels)} patients")
print("Sample labels:")
for i, (key, value) in enumerate(labels.items()):
    if i < 5:  # Show first 5 entries
        print(f"{key}: {value}")

# Save as .pt file
#output_path = "/home/jakaria/final_classification_dataset_femur/labels/labels.pt"
output_path = "/home/jakaria/final_classification_dataset_femur_original/all_mesh/sdf_data/SdfSamples/scaled_obj_files/labels.pt"
torch.save(labels, output_path)
print(f"\nLabels saved to: {output_path}")

# Verify the saved file
loaded_labels = torch.load(output_path)
print(f"\nVerification - loaded {len(loaded_labels)} labels")
print("Sample loaded labels:")
for i, (key, value) in enumerate(loaded_labels.items()):
    if i < 3:
        print(f"{key}: {value}")

Created labels for 116 patients
Sample labels:
9008561: 0.0
9013798: 0.0
9017909: 0.0
9036770: 0.0
9036948: 0.0

Labels saved to: /home/jakaria/final_classification_dataset_femur_original/all_mesh/sdf_data/SdfSamples/scaled_obj_files/labels.pt

Verification - loaded 116 labels
Sample loaded labels:
9008561: 0.0
9013798: 0.0
9017909: 0.0


In [4]:
test_labels = torch.load(output_path)
print(f"\nVerification - loaded {len(test_labels)} labels")
print("Sample loaded labels:")
for i, (key, value) in enumerate(test_labels.items()):
    if i > 100:
        print(f"{key}: {value}")


Verification - loaded 116 labels
Sample loaded labels:
9781749: 1.0
9854269: 0.0
9858216: 1.0
9876530: 0.0
9878765: 0.0
9879069: 0.0
9895555: 1.0
9907090: 0.0
9916140: 0.0
9933836: 1.0
9943227: 1.0
9967815: 0.0
9973322: 0.0
9978579: 0.0
9988421: 0.0


In [16]:
import torch
labels_path = '../../../hippocampus_data_tle_ms_age_and_0_1/hippo_ms_label_age_32_71/labels.pt'
labels = torch.load(labels_path)
print(f"Labels: {labels}")
print(labels['ab300_001'])

Labels: {'ab300_001': array([0.34117647, 0.        ]), 'ab300_003': array([0.41176471, 0.        ]), 'ab300_004': array([0.56470588, 0.        ]), 'ab300_007': array([0.61176471, 0.        ]), 'ab300_008': array([0.32941176, 0.        ]), 'ab300_009': array([0.70588235, 0.        ]), 'ab300_010': array([0.35294118, 0.        ]), 'ab300_011': array([0.72941176, 0.        ]), 'ab300_015': array([0.34117647, 0.        ]), 'ab300_016': array([0.62352941, 0.        ]), 'ab300_021': array([0.56470588, 0.        ]), 'ab300_022': array([0.63529412, 0.        ]), 'ab300_024': array([0.42352941, 0.        ]), 'ab300_026': array([0.51764706, 0.        ]), 'ab300_027': array([0.50588235, 0.        ]), 'ab300_029': array([0.71764706, 0.        ]), 'ab300_031': array([0.50588235, 0.        ]), 'ab300_033': array([0.43529412, 0.        ]), 'ab300_035': array([0.65882353, 0.        ]), 'ab300_036': array([0.76470588, 0.        ]), 'ab300_037': array([0.32941176, 0.        ]), 'ab300_038': array([0.352

In [15]:
import torch
import re

# Load the original labels
labels_path = '../../../hippocampus_data_tle_ms_age_and_0_1/hippo_ms_label_age_32_71/labels.pt'
labels = torch.load(labels_path)

print("Original labels keys (first 10):")
print(list(labels.keys())[:10])

# Create a new dictionary with updated keys
updated_labels = {}
for old_key, value in labels.items():
    # Convert patterns like ms1, ms10, etc. to ms_1, ms_10, etc.
    if old_key.startswith('ms') and re.match(r'ms\d+', old_key):
        new_key = re.sub(r'ms(\d+)', r'ms_\1', old_key)
        updated_labels[new_key] = value
        print(f"Updated: {old_key} -> {new_key}")
    else:
        # Keep the original key if it doesn't match the pattern
        updated_labels[old_key] = value

# Define regex pattern outside f-string
ms_pattern = r'ms\d+'
ms_keys_count = len([k for k in labels.keys() if k.startswith('ms') and re.match(ms_pattern, k)])
print(f"\nTotal keys updated: {ms_keys_count}")
print("Updated labels keys (first 10):")
print(list(updated_labels.keys())[:10])

# Save the updated labels back to the file
torch.save(updated_labels, labels_path)
print(f"\nUpdated labels saved to {labels_path}")

# Verify the changes
labels_updated = torch.load(labels_path)
print(f"\nVerification - sample updated key: {list(labels_updated.keys())[0]}")

Original labels keys (first 10):
['ab300_001', 'ab300_003', 'ab300_004', 'ab300_007', 'ab300_008', 'ab300_009', 'ab300_010', 'ab300_011', 'ab300_015', 'ab300_016']
Updated: ms1 -> ms_1
Updated: ms2 -> ms_2
Updated: ms3 -> ms_3
Updated: ms4 -> ms_4
Updated: ms6 -> ms_6
Updated: ms7 -> ms_7
Updated: ms8 -> ms_8
Updated: ms9 -> ms_9
Updated: ms10 -> ms_10
Updated: ms11 -> ms_11
Updated: ms12 -> ms_12
Updated: ms13 -> ms_13
Updated: ms14 -> ms_14
Updated: ms15 -> ms_15
Updated: ms16 -> ms_16
Updated: ms17 -> ms_17
Updated: ms18 -> ms_18
Updated: ms19 -> ms_19
Updated: ms20 -> ms_20
Updated: ms21 -> ms_21
Updated: ms22 -> ms_22
Updated: ms23 -> ms_23
Updated: ms24 -> ms_24
Updated: ms25 -> ms_25
Updated: ms26 -> ms_26
Updated: ms27 -> ms_27
Updated: ms28 -> ms_28
Updated: ms29 -> ms_29
Updated: ms30 -> ms_30
Updated: ms31 -> ms_31
Updated: ms32 -> ms_32
Updated: ms33 -> ms_33
Updated: ms34 -> ms_34
Updated: ms35 -> ms_35
Updated: ms36 -> ms_36
Updated: ms37 -> ms_37
Updated: ms38 -> ms_38
U